In [1]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-04-30 16:29:37,276 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-04-30 16:29:38,078 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-04-30 16:29:38,433 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


In [2]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

def get_negative_example_sentences(df):
    # Find all unique texts
    unique_texts = df["text"].unique()

    # Split unique texts into sentences
    all_sentences_from_text = set()
    for text in unique_texts:
        doc = nlp(text)
        all_sentences_from_text.update([sent.text for sent in doc.sents])

    # Set of sentences already in the 'sentence' column
    existing_sentences = set(df["sentence"])

    # Find sentences that are not in the 'sentence' column
    new_sentences = all_sentences_from_text - existing_sentences

    return new_sentences


def filter_job_ads(labelled_df):
    job_ids = labelled_df["id"].unique()

    # skip the first 10 job ads - we didn't know what we were labelling at that point
    target_ids = job_ids[10:]

    labelled_df_clean = labelled_df[labelled_df["id"].isin(target_ids)]
    # get rid of empty spans
    labelled_df_clean = labelled_df_clean[labelled_df_clean["span"] != ""]
    return labelled_df_clean

In [3]:
labelled_sents = get_labelled_job_sentences()[0]

labelled_data = pdu.get_spans_and_sentences(labelled_sents)

labelled_df = pd.DataFrame(columns=["span", "sent", "text", "job_id"])

for key in labelled_data.keys():
    temp_df = pd.DataFrame(labelled_data[key])
    temp_df["id"] = int(key)
    labelled_df = pd.concat([labelled_df, temp_df])

labelled_df = labelled_df.drop(["job_id"], axis=1)

labelled_df_clean = filter_job_ads(labelled_df)

labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)

negative_sentences = get_negative_example_sentences(labelled_df_clean)

negative_df = pd.DataFrame(list(negative_sentences))
negative_df["label"] = 0
negative_df.columns = ["span", "label"]

positive_df = labelled_df_clean[["span"]]
positive_df["label"] = 1

training_ml_df = pd.concat([positive_df, negative_df])

# Splitting the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
        training_ml_df.drop(["label"], axis=1),
        training_ml_df["label"],
        test_size=0.4,
        random_state=SEED,
    )
X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED
    )

2024-04-30 16:29:39,541 - dap_job_quality - INFO - File job_quality/prodigy/binary_classifier_labelled_data/20240416/job_sentences_labelled_20240416.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240416.jsonl


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/1212298874.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_df["label"] = 1


In [4]:
input_sentences = X_train['span'].tolist()

In [5]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v3.csv")
lookup.head()

,dimension,subcategory,target_phrase,Notes
0,pay and benefits,COMP,pension,NaN
1,pay and benefits,COMP,bonus,NaN
2,pay and benefits,COMP,salary,NaN
3,pay and benefits,COMP,compensation,NaN
4,pay and benefits,COMP,pay,NaN


In [6]:
targets = lookup['target_phrase'].tolist()

In [7]:
target_embeddings = model.encode(targets, show_progress_bar=True)

Batches: 100%|██████████| 2/2 [00:00<00:00,  6.86it/s]


In [8]:
lookup['embeddings'] = target_embeddings.tolist()

In [9]:
lookup.head()

,dimension,subcategory,target_phrase,Notes,embeddings
0,pay and benefits,COMP,pension,NaN,"[0.016609827056527138, 0.15209849178791046, -0..."
1,pay and benefits,COMP,bonus,NaN,"[-0.012775829061865807, 0.027487050741910934, ..."
2,pay and benefits,COMP,salary,NaN,"[-0.03451469913125038, 0.05973529443144798, -0..."
3,pay and benefits,COMP,compensation,NaN,"[-0.11117522418498993, 0.11772848665714264, 0...."
4,pay and benefits,COMP,pay,NaN,"[-0.06232767552137375, 0.09840542823076248, 0...."


In [23]:
def get_n_most_similar_phrases(input_sentence, 
                                 lookup,
                                 model,
                                 n: int = 5):
    most_similar_phrases = {}
    
    input_embedding = model.encode(input_sentence)
    
    similarities = [cosine_similarity([input_embedding], [embed])[0][0] for embed in lookup['embeddings'].apply(pd.Series).values]
    
    top_indices = np.argsort(similarities)[::-1][:n]
    
    similar_phrases = lookup.iloc[top_indices]
    similar_phrases['similarity'] = [similarities[i] for i in top_indices]
    
    most_similar_phrases[input_sentence] = similar_phrases[['dimension', 'subcategory', 'target_phrase', 'similarity']]
        
    return most_similar_phrases

In [11]:
input_sentences[0:10]

['You will be required support the Agency Services Team ensuring the  consistent delivery of exceptional customer service to existing Financial Advisers, and manage the setup of any new Agency or firm within the Society They have experienced some impressive growth recently, and have a leadership team in place to make sure this continues and have big plans for the future, meaning there is some great opportunities at the moment for hard working and career minded people who  want to join them on this journey.',
 '£30,000 - £35,000K (+ bonuses of up to 90%)',
 'Up to £28,000',
 'Work with our wonderful clients and support them in their hiring plans.',
 'Brand new fully kitted out offices, minutes from the subway station',
 'Structured career development opportunities',
 'Are you a Psychology graduate with aspirations to become a psychologist, Counsellor or Therapist?',
 'Company pension',
 'Such as all equipment, passes, inductions etc.',
 'If you wish to discuss this role in more detail, 

In [34]:
for sentence in input_sentences[0:10]:
    print(f"Sentence: {sentence}")
    print(get_n_most_similar_phrases(sentence, lookup, model))

Sentence: You will be required support the Agency Services Team ensuring the  consistent delivery of exceptional customer service to existing Financial Advisers, and manage the setup of any new Agency or firm within the Society They have experienced some impressive growth recently, and have a leadership team in place to make sure this continues and have big plans for the future, meaning there is some great opportunities at the moment for hard working and career minded people who  want to join them on this journey.


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.36it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/4230199457.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'You will be required support the Agency Services Team ensuring the  consistent delivery of exceptional customer service to existing Financial Advisers, and manage the setup of any new Agency or firm within the Society They have experienced some impressive growth recently, and have a leadership team in place to make sure this continues and have big plans for the future, meaning there is some great opportunities at the moment for hard working and career minded people who  want to join them on this journey.':                         dimension  subcategory       target_phrase  similarity
32  job design and nature of work       CAREER      career advance    0.316831
33  job design and nature of work       CAREER  career progression    0.297887
30  job design and nature of work          L&D            training    0.215675
11               pay and benefits  SPONSORSHIP    visa sponsorship    0.214118
21               employment terms        HOURS           full time    0.199142}
Sentence: £

Batches: 100%|██████████| 1/1 [00:00<00:00, 81.61it/s]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/4230199457.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]


{'£30,000 - £35,000K (+ bonuses of up to 90%)':            dimension subcategory target_phrase  similarity
2   pay and benefits        COMP        salary    0.595485
5   pay and benefits        COMP     per annum    0.526548
0   pay and benefits        COMP       pension    0.394335
3   pay and benefits        COMP  compensation    0.387925
39  pay and benefits       PERKS     Discounts    0.321387}
Sentence: Up to £28,000


...

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/4230199457.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]
Batches: 100%|██████████| 1/1 [00:00<00:00, 115.02it/s]


{'Such as all equipment, passes, inductions etc.':                         dimension subcategory                  target_phrase  \
30  job design and nature of work         L&D                       training   
15              work life balance  FLEX_HOURS       flexible working options   
31  job design and nature of work         L&D         learning & development   
14              work life balance  FLEX_HOURS  flexible working arrangements   
6                pay and benefits        COMP                       overtime   

    similarity  
30    0.269619  
15    0.197547  
31    0.184698  
14    0.170671  
6     0.151845  }
Sentence: If you wish to discuss this role in more detail, please contact Kieran Boyle at CKB Recruitment - General Insurance and Financial Services Recruitment Specialists.


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/4230199457.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]
Batches: 100%|██████████| 1/1 [00:00<00:00, 78.14it/s]

{'If you wish to discuss this role in more detail, please contact Kieran Boyle at CKB Recruitment - General Insurance and Financial Services Recruitment Specialists.':                         dimension subcategory       target_phrase  similarity
16              work life balance  FLEX_HOURS           job share    0.310983
17              work life balance  FLEX_HOURS           job share    0.310983
34               pay and benefits       PERKS      Life insurance    0.292872
35               pay and benefits       PERKS  Private healthcare    0.281346
32  job design and nature of work      CAREER      career advance    0.277113}



/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_95222/4230199457.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  similar_phrases['similarity'] = [similarities[i] for i in top_indices]
